Pydantic to structure Gemini outputs

In [48]:
from dotenv import load_dotenv
#import os
from google import genai

#load_dotenv()

client = genai.Client()

response = client.models.generate_content(
    model="gemini-2.5-flash", contents="Tell me a programming joke"
)
response

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""Why do programmers prefer dark mode?

Because light attracts bugs!

---

Or, for a classic:

"Knock, knock."
"Who's there?"
*(You hear two different voices simultaneously say different things)*
"Race condition!""""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='epi-aI2bFrnyxs0Ps7DsqQU',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=53,
    prompt_token_count=6,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=6
      ),
    ],
    thoughts_token_count=1186,
    total_token_count=1245
  )
)

In [49]:
print(response.text)

Why do programmers prefer dark mode?

Because light attracts bugs!

---

Or, for a classic:

"Knock, knock."
"Who's there?"
*(You hear two different voices simultaneously say different things)*
"Race condition!"


In [50]:
def ask_llm(prompt):
    response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)
    
    return response.text

ask_llm("Du är en Göteborgare, ge mig ett gör gött göteborgs skämt ;D")

"Jomenvisst, här ska du få ett som e så gött så du får ta å hålla i dig!\n\nDet va två göteborgare som satt å tjöta, å så säger den ena:\n– Nämen du Lärs, nu måste du la ändå fatta!\nDå glor Lärs på'n å säger:\n– Fatta? Fatta va? Varför skulle ja fälla'n då, den står ju rätt upp å ner ju!\n\nÄr inte det gött så säg! Ha en fin dag la! ;D"

# Try to get data from our LLM

In [51]:
response = ask_llm(
    """
        Du är en expert inom köp och sälj av bostäder, likt en proffsig mäklare.   
        Generera bostadspriser, månadsavgifter, adress, boarea & stad i json-format(ej markdown)
            
        Exempel:
                {
                   "adress": "Fågelvägen 5,
                   "price_sek":3000000,
                   "city":"Göteborg",
                   "monthly_fee":4000,
                   "area":60
                }
        
        Ge mig en lista på 5 bostäder
                   
"""
)
print(response)

[
   {
      "adress": "Vasagatan 12A",
      "price_sek": 5950000,
      "city": "Stockholm",
      "monthly_fee": 3850,
      "area": 75
   },
   {
      "adress": "Linnegatan 45B",
      "price_sek": 3795000,
      "city": "Göteborg",
      "monthly_fee": 4200,
      "area": 68
   },
   {
      "adress": "Davidshallsgatan 3, lgh 1201",
      "price_sek": 2850000,
      "city": "Malmö",
      "monthly_fee": 3100,
      "area": 55
   },
   {
      "adress": "Kungsängsesplanaden 7",
      "price_sek": 4100000,
      "city": "Uppsala",
      "monthly_fee": 4900,
      "area": 88
   },
   {
      "adress": "Björkvägen 15",
      "price_sek": 3495000,
      "city": "Västerås",
      "monthly_fee": 0,
      "area": 115
   }
]


## Parse data and validate data

In [52]:
from pydantic import BaseModel, Field
import json

class Apartment(BaseModel):
    adress: str
    price_sek: int = Field(gt = 1000000, lt = 8000000)
    city: str
    monthly_fee: int
    area: int

class ApartmentList(BaseModel):
    objects: list[Apartment]

apartments = ApartmentList.model_validate({"objects": json.loads(response)})

print(apartments)

objects=[Apartment(adress='Vasagatan 12A', price_sek=5950000, city='Stockholm', monthly_fee=3850, area=75), Apartment(adress='Linnegatan 45B', price_sek=3795000, city='Göteborg', monthly_fee=4200, area=68), Apartment(adress='Davidshallsgatan 3, lgh 1201', price_sek=2850000, city='Malmö', monthly_fee=3100, area=55), Apartment(adress='Kungsängsesplanaden 7', price_sek=4100000, city='Uppsala', monthly_fee=4900, area=88), Apartment(adress='Björkvägen 15', price_sek=3495000, city='Västerås', monthly_fee=0, area=115)]


In [53]:
apartments.objects


[Apartment(adress='Vasagatan 12A', price_sek=5950000, city='Stockholm', monthly_fee=3850, area=75),
 Apartment(adress='Linnegatan 45B', price_sek=3795000, city='Göteborg', monthly_fee=4200, area=68),
 Apartment(adress='Davidshallsgatan 3, lgh 1201', price_sek=2850000, city='Malmö', monthly_fee=3100, area=55),
 Apartment(adress='Kungsängsesplanaden 7', price_sek=4100000, city='Uppsala', monthly_fee=4900, area=88),
 Apartment(adress='Björkvägen 15', price_sek=3495000, city='Västerås', monthly_fee=0, area=115)]

In [54]:
apartments.objects[1].adress, apartments.objects[1].city

('Linnegatan 45B', 'Göteborg')

In [55]:
adresses = [apartment.adress for apartment in apartments.objects]

adresses

['Vasagatan 12A',
 'Linnegatan 45B',
 'Davidshallsgatan 3, lgh 1201',
 'Kungsängsesplanaden 7',
 'Björkvägen 15']

In [70]:
adresses3 = [
    apartment.adress
    for apartment in apartments.objects
    if apartment.price_sek < 3000000
]

adresses3

['Davidshallsgatan 3, lgh 1201']

get adress, city, price, monthly_fee for the interval between 4m - 8m

In [74]:
#adresses2 = [
#    apartment.adress
#    for apartment in apartments.objects
#    if apartment.price_sek >4000000 and apartment.price_sek < 8000000
#]

# or like this, this way is preferable since its less redundant and still easy to understand
adresses_4_to_8 = [
    apartment.adress
    for apartment in apartments.objects
    if 4000000 < apartment.price_sek < 8000000
]

adresses_4_to_8

['Vasagatan 12A', 'Kungsängsesplanaden 7']

In [77]:
adresses_4_to_8 = [
    [home.adress, home.city, home.price_sek, home.monthly_fee]
    for home 
    in apartments.objects
    if 4000000 < home.price_sek < 8000000
]
adresses_4_to_8

[['Vasagatan 12A', 'Stockholm', 5950000, 3850],
 ['Kungsängsesplanaden 7', 'Uppsala', 4100000, 4900]]

### convert to DataFrame

In [78]:
import pandas as pd

# Filter
filtered_homes = [
    home for home in apartments.objects if 4_000_000 < home.price_sek < 8_000_000
]

filtered_homes

[Apartment(adress='Vasagatan 12A', price_sek=5950000, city='Stockholm', monthly_fee=3850, area=75),
 Apartment(adress='Kungsängsesplanaden 7', price_sek=4100000, city='Uppsala', monthly_fee=4900, area=88)]

In [79]:
# Convert to df
df_homes_filtered = pd.DataFrame(
    [
        home.model_dump(include={"address", "city", "price_sek", "monthly_fee"})
        for home in filtered_homes
    ]
)

df_homes_filtered

,price_sek,city,monthly_fee
0,5950000,Stockholm,3850
1,4100000,Uppsala,4900


In [81]:
df_homes_filtered.to_csv("filtered_homes.csv", index=False)

# save to json - serialize pydantic model

In [82]:
apartments.model_dump()

{'objects': [{'adress': 'Vasagatan 12A',
   'price_sek': 5950000,
   'city': 'Stockholm',
   'monthly_fee': 3850,
   'area': 75},
  {'adress': 'Linnegatan 45B',
   'price_sek': 3795000,
   'city': 'Göteborg',
   'monthly_fee': 4200,
   'area': 68},
  {'adress': 'Davidshallsgatan 3, lgh 1201',
   'price_sek': 2850000,
   'city': 'Malmö',
   'monthly_fee': 3100,
   'area': 55},
  {'adress': 'Kungsängsesplanaden 7',
   'price_sek': 4100000,
   'city': 'Uppsala',
   'monthly_fee': 4900,
   'area': 88},
  {'adress': 'Björkvägen 15',
   'price_sek': 3495000,
   'city': 'Västerås',
   'monthly_fee': 0,
   'area': 115}]}

In [84]:
with open("apartments.json", "w") as json_file:
    json_file.write(apartments.model_dump_json(indent=3))

# pandas dataframe another way

In [86]:
apartments.objects

[Apartment(adress='Vasagatan 12A', price_sek=5950000, city='Stockholm', monthly_fee=3850, area=75),
 Apartment(adress='Linnegatan 45B', price_sek=3795000, city='Göteborg', monthly_fee=4200, area=68),
 Apartment(adress='Davidshallsgatan 3, lgh 1201', price_sek=2850000, city='Malmö', monthly_fee=3100, area=55),
 Apartment(adress='Kungsängsesplanaden 7', price_sek=4100000, city='Uppsala', monthly_fee=4900, area=88),
 Apartment(adress='Björkvägen 15', price_sek=3495000, city='Västerås', monthly_fee=0, area=115)]

In [91]:
adresses = [apartment.adress for apartment in apartments.objects]
prices = [apartment.price_sek for apartment in apartments.objects]
monthly_fees = [apartment.monthly_fee for apartment in apartments.objects]
areas = [apartment.area for apartment in apartments.objects]
df = pd.DataFrame(
    {"adress": adresses, "area": areas, "price": prices, "monthly_fee": monthly_fees}
)
df

,adress,area,price,monthly_fee
0,Vasagatan 12A,75,5950000,3850
1,Linnegatan 45B,68,3795000,4200
2,"Davidshallsgatan 3, lgh 1201",55,2850000,3100
3,Kungsängsesplanaden 7,88,4100000,4900
4,Björkvägen 15,115,3495000,0


## put this data into a duckdb dataframe

Approach 1
- duckdb read_csv_auto...

Approach 2
- open up a connection to duckdb
- create schema
- create tables from df

Approach 3 (naive)
- create tables
- insert data manually (or parse with python)

Approach 4 - dlt
- extract and load df into duckdb


In [97]:
import duckdb
import dlt

@dlt.resource(write_disposition="replace", table_name="apartment")
def load_data():
    return df

pipeline = dlt.pipeline(
    pipeline_name="apartments",
    destination="duckdb",
    dataset_name="staging"
)

load_info = pipeline.run(load_data())
print(load_info)

Pipeline apartments load step completed in 1.19 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\edwin\Documents\AI_Engineering_Course_Material\AI_Engineering_Edwin_Lindblom\code_alongs\07b_pydantic_gemini\apartments.duckdb location to store data
Load package 1757331560.2534232 is LOADED and contains no failed jobs


In [98]:
import duckdb
import dlt

@dlt.resource(write_disposition="replace", table_name="apartment")
def load_data():
    for record in df.to_dict(orient="records"):
        yield record

pipeline = dlt.pipeline(
    pipeline_name="apartments",
    destination="duckdb",
    dataset_name="staging"
)

load_info = pipeline.run(load_data())
print(load_info)

Pipeline apartments load step completed in 0.20 seconds
1 load package(s) were loaded to destination duckdb and into dataset staging
The duckdb destination used duckdb:///c:\Users\edwin\Documents\AI_Engineering_Course_Material\AI_Engineering_Edwin_Lindblom\code_alongs\07b_pydantic_gemini\apartments.duckdb location to store data
Load package 1757331794.7060814 is LOADED and contains no failed jobs
